# Political Regime and Democracy Index Visualizations (Data2)

This notebook provides diverse visualizations for the regime types and democracy indices dataset.

## Required Visualizations:
1. **Line Chart**: Evolution of democracy index for major countries.
2. **Map**: Global choropleth map of regime types.
3. **Bar Chart**: Average democracy index by regime type.
4. **Table**: Top 10 countries with the highest democracy scores in 2020.
5. **Heatmap**: Correlation matrix of political indices.
6. **Box Plot**: Distribution of democracy scores across all countries.

### 1. Line Chart: Evolution of Democracy Index
Tracking `v2x_polyarchy` for a selection of countries over time.

In [1]:
import os
import pandas as pd
import plotly.express as px

# Save plots under the workspace-level figures folder
base_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
output_dir = os.path.join(base_dir, 'figures', 'data2')
os.makedirs(output_dir, exist_ok=True)

def save_fig(fig, name):
    path = os.path.join(output_dir, name)
    fig.write_image(path + '.png')

# Load the dataset
data2_path = 'ert.csv'
df2 = pd.read_csv(data2_path)

print("Data loaded successfully.")
df2.head()

Data loaded successfully.


,Unnamed: 0,country_id,country_text_id,country_name,year,v2x_regime,v2x_polyarchy,v2x_polyarchy_codelow,v2x_polyarchy_codehigh,reg_start_year,...,aut_ep_start_year,aut_ep_end_year,aut_pre_ep_year,aut_ep_termination,aut_ep_prch,aut_ep_pbr,aut_ep_subreg,aut_ep_outcome,aut_ep_outcome_agg,aut_ep_censored
0,1,3,MEX,Mexico,1900,0.0,0.124,0.110,0.134,1900.0,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0,0,0
1,2,3,MEX,Mexico,1901,0.0,0.108,0.094,0.119,1900.0,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0,0,0
2,3,3,MEX,Mexico,1902,0.0,0.108,0.094,0.119,1900.0,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0,0,0
3,4,3,MEX,Mexico,1903,0.0,0.108,0.094,0.119,1900.0,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0,0,0
4,5,3,MEX,Mexico,1904,0.0,0.108,0.094,0.119,1900.0,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0,0,0


In [2]:
countries = ['United States', 'United Kingdom', 'India', 'China', 'Brazil']
df_filtered = df2[df2['country_name'].isin(countries)]

fig_line2 = px.line(df_filtered, x='year', y='v2x_polyarchy', color='country_name',
                   title='Democracy Index (v2x_polyarchy) Trends')
fig_line2.show()
# save
save_fig(fig_line2, 'democracy_trends_selected_countries')

### 2. Map: Global Choropleth of Regime Types
Visualizing the distribution of regime types (`v2x_regime`) for the most recent year.

In [3]:
latest_year2 = df2['year'].max()
df_latest2 = df2[df2['year'] == latest_year2]

fig_map2 = px.choropleth(df_latest2, locations="country_text_id",
                        color="v2x_regime",
                        hover_name="country_name",
                        title=f"Global Political Regimes in {latest_year2}",
                        color_continuous_scale=px.colors.sequential.Viridis)
fig_map2.show()
# save
save_fig(fig_map2, 'choropleth_regimes')

### 3. Bar Chart: Average Democracy Index by Regime Type
Understanding the relationship between regime classification and the polyarchy score.

In [4]:
# Bar: average democracy index by regime
avg_poly = df_latest2.groupby('v2x_regime', as_index=False)['v2x_polyarchy'].mean()
fig_bar2 = px.bar(avg_poly, x='v2x_regime', y='v2x_polyarchy',
                  title='Average Democracy Index by Regime Type',
                  labels={'v2x_regime':'Regime Type','v2x_polyarchy':'Average polyarchy'})
fig_bar2.show()
# save
save_fig(fig_bar2, 'avg_democracy_by_regime')

### 4. Table: Top 10 Countries by Democracy Score (2020)
Identifying the most democratic nations in 2020.

In [5]:
df_2020 = df2[df2['year'] == 2020]
top_10_dem = df_2020.nlargest(10, 'v2x_polyarchy')[['country_name', 'v2x_polyarchy', 'v2x_regime']].sort_values('v2x_polyarchy', ascending=True)

fig_table2 = px.bar(
    top_10_dem,
    x='v2x_polyarchy',
    y='country_name',
    orientation='h',
    color='v2x_regime',
    color_continuous_scale='Viridis',
    title='Top 10 Most Democratic Countries in 2020',
    labels={
        'country_name': 'Country',
        'v2x_polyarchy': 'Democracy score (v2x_polyarchy)',
        'v2x_regime': 'Regime score'
    }
)

fig_table2.update_layout(
    yaxis=dict(autorange='reversed'),
    margin=dict(l=20, r=20, t=60, b=20),
    height=520
)
fig_table2.show()
# save
save_fig(fig_table2, 'top10_democracy_2020')

### 5. Heatmap: Correlation Matrix of Indices
Exploring how different political indices relate to each other.

In [6]:
# Heatmap: correlation matrix of available political indices
heat_cols = [
    'v2x_polyarchy',
    'v2x_regime',
    'v2x_polyarchy_codelow',
    'v2x_polyarchy_codehigh',
    'reg_trans',
]
heat_cols = [col for col in heat_cols if col in df2.columns]

corr = df2[heat_cols].corr(numeric_only=True).round(2)
fig_heat = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale='Viridis',
    zmin=-1,
    zmax=1,
    aspect='auto',
    title='Correlation Matrix of Political Indices'
)
fig_heat.update_layout(
    width=900,
    height=700,
    margin=dict(l=60, r=30, t=80, b=60),
    coloraxis_colorbar=dict(title='Corr')
)
fig_heat.update_xaxes(side='top', tickangle=-35)
fig_heat.update_yaxes(autorange='reversed')
fig_heat.show()
# save
save_fig(fig_heat, 'correlation_heatmap')

### 6. Box Plot: Distribution of Democracy Scores
Showing the variance and outliers of the Democracy Index for the latest year.

In [7]:
fig_box = px.box(df2, x='v2x_regime', y='v2x_polyarchy',
                 title='Distribution of Democracy Scores by Regime Type')
fig_box.show()
# save
save_fig(fig_box, 'box_democracy_by_regime')